In [1]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

In [12]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# 1. Load the variables from .env into the environment
load_dotenv() 

# 2. Retrieve the token securely
my_token = os.getenv("HF_TOKEN")

# 3. Log in without exposing the key in the code
login(token=my_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


#### Select the model to use

In [7]:
model_id = "meta-llama/Llama-3.2-1B"

#### Configure quantization

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

In [14]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False
)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B.
403 Client Error. (Request ID: Root=1-69516897-6349aa1467c3ddfb7df08b8e;f7ba81a9-c8a5-4359-b068-e5d501f42d03)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/config.json.
Your request to access model meta-llama/Llama-3.2-1B is awaiting a review from the repo authors.

#### Load the tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Fix for Llama 3
tokenizer.padding_side = "right" # Trainer expects right padding

: 

#### Prepare for training

In [ ]:
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

#### Tokenize the dataset

In [ ]:
# Load a raw dataset
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

# Define the max context length 
MAX_LENGTH = 1024 

In [ ]:


def format_and_tokenize(example):
    """
    1. Apply the Chat Template (User/Assistant format)
    2. Tokenize the result
    3. Create labels (copy of input_ids)
    4. Mask padding tokens in labels
    """
    
    # A. Format the text using Llama 3's official chat template
    # The dataset column is 'text', but usually we need to structure it as messages.
    # Since this specific dataset is already formatted in strings, we can just tokenize.
    # BUT, if you had raw data, you would structure it like this:
    # messages = [
    #    {"role": "user", "content": example['question']},
    #    {"role": "assistant", "content": example['answer']}
    # ]
    # text = tokenizer.apply_chat_template(messages, tokenize=False)

    text = example['text'] # Assuming raw text for this demo

    # B. Tokenize
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length", # Pad to fixed length
    )

    # C. Create Labels
    # In Causal LM, the input is the label (self-supervised).
    # We copy input_ids to labels.
    input_ids = tokenized["input_ids"]
    labels = input_ids[:] # Create a copy

    # D. Handle Padding
    # We must replace the padding token IDs in the 'labels' with -100
    # so the loss function ignores them.
    pad_token_id = tokenizer.pad_token_id
    for i, token_id in enumerate(labels):
        if token_id == pad_token_id:
            labels[i] = -100
    
    tokenized["labels"] = labels
    return tokenized

# Apply the function to the dataset
tokenized_dataset = dataset.map(format_and_tokenize)

# Remove the raw text columns, we only need tensors now
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

#### Data collector

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#### Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./manual_llama_finetune",
    per_device_train_batch_size=1, # Keep small for GPU
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    max_steps=100, # Short run for demonstration
    save_strategy="no",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    # tokenizer is passed here mainly for saving it later, 
    # the processing is already done
    tokenizer=tokenizer, 
)

In [ ]:
trainer.train()